Подключение Google Disk, создание итоговой директории, куда положить csv таблицу:

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")
# Заменить путь нужный
drive_results_path = "/content/drive/MyDrive/pyannote_results"
os.makedirs(drive_results_path, exist_ok=True)

Скачиваем необходимые зависимости:

In [ ]:
!apt-get update && apt-get install -y ffmpeg

!pip install --no-cache-dir \
    "pyannote.audio" \
    "pydub" \
    "librosa" \
    "datasets" \
    "huggingface_hub"

Определяем девайс, на котором будет исполняться сложный код: CPU, GPU (cuda)

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Selected device: {device}")

Обязательно авторизируемся в Hugging Face Hub для возможности скачивания модели диаризации:

In [ ]:
from google.colab import userdata
from huggingface_hub import login
import os

try:
    print("Logging in HuggingFace...")
    hf_token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    print("Token has downloaded from Colab's secrets!")
    login(token=hf_token)
    print("Login successfully!")
except Exception as e:
    print(f"Error: {e}")
    hf_token = None

Определяем допуски для метрики DER:

In [ ]:
from pyannote.metrics.diarization import DiarizationErrorRate, DiarizationPurity, DiarizationCoverage
from pyannote.metrics.detection import DetectionErrorRate

configs = {
    'strict': {'collar': 0.0, 'skip_overlap': False},
    'collar': {'collar': 0.25, 'skip_overlap': False},
    'no_ovl': {'collar': 0.0, 'skip_overlap': True},
    'clean':  {'collar': 0.25, 'skip_overlap': True}
}

metrics_vault = {}
for c_name, params in configs.items():
    metrics_vault[c_name] = {
        'der': DiarizationErrorRate(**params),
        'purity': DiarizationPurity(**params),
        'coverage': DiarizationCoverage(**params),
        'det': DetectionErrorRate(collar=params['collar'])
    }

На тестовых данных из датасета AMI (ihm) тестируем базовый пайплайн Pyannote. В конце сохраняем все данные в csv таблицу, которая будет состоять из общей ошибки DER, трех ее составляющих: Missed, False alarm, Confusion и из метрик Purity и Coverage.

In [ ]:
from datasets import load_dataset
from datetime import datetime
import os
from pyannote.core import Annotation, Segment
from pyannote.audio import Pipeline
import pandas as pd
import soundfile as sf
import torch

diarization_data_set = "diarizers-community/ami"
dataset_config = "ihm" # default: None

os.makedirs("dataset_samples", exist_ok=True)
os.makedirs("results", exist_ok=True)

pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1")
pipeline.to(torch.device(device))
dataset = load_dataset(f"{diarization_data_set}", name=dataset_config, split="test", streaming=True)

test_samples = list(dataset)
results = []
for idx, sample in enumerate(test_samples):
    audio = sample["audio"]
    audio_path = f"dataset_samples/sample_{idx}.wav"
    sf.write(audio_path, audio["array"], audio["sampling_rate"])

    reference = Annotation()
    for start, end, speaker in zip(sample['timestamps_start'], sample['timestamps_end'], sample['speakers']):
        segment = Segment(start, end)
        reference[segment] = speaker

    hypothesis = pipeline(audio_path)
    hypothesis_diarization = hypothesis.speaker_diarization

    res = {'file': f"sample_{idx}"}
    for c_name, m_group in metrics_vault.items():
        res[f'DER_{c_name}'] = m_group['der'](reference, hypothesis_diarization)
        res[f'Purity_{c_name}'] = m_group['purity'](reference, hypothesis_diarization)
        res[f'Coverage_{c_name}'] = m_group['coverage'](reference, hypothesis_diarization)
        res[f'DetER_{c_name}'] = m_group['det'](reference, hypothesis_diarization)

        components = m_group['der'].compute_components(reference, hypothesis_diarization)
        res[f'FA_{c_name}'] = components['false alarm']
        res[f'Miss_{c_name}'] = components['missed detection']
        res[f'Conf_{c_name}'] = components['confusion']
        res[f'Total_Speech_{c_name}'] = components['total']

        print("-" * 60)
        print("DER: ", res[f'DER_{c_name}'])
        print("Purity: ", res[f'Purity_{c_name}'])
        print("Coverage: ", res[f'Coverage_{c_name}'])
        print("DetER: ", res[f'DetER_{c_name}'])
        print(f"- {c_name} -")
        print("FA: ", res[f'FA_{c_name}'])
        print("Miss:", res[f'Miss_{c_name}'])
        print("Conf: ", res[f'Conf_{c_name}'])
    print("================================================== New Test audio file ==================================================")

    results.append(res)

current_time = datetime.now().strftime("%Y%m%d_%H%M%S")
df = pd.DataFrame(results)
df.to_csv(f"results/pyannote_metrics_{current_time}.csv", index=False)
if not os.path.exists(f"{drive_results_path}"):
    os.makedirs(f"{drive_results_path}/")
    df.to_csv(f"{drive_results_path}/pyannote_metrics_{current_time}.csv", index=False)

Визуализируем результаты: общие (средняя ошибка DER) и для каждой записи. Здесь визуализируется DER из 3 частей и отдельным графиком Purity и Coverage.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mpl_toolkits.axisartist.axislines import Subplot
from matplotlib.transforms import blended_transform_factory


def plot_single_histogram(data, categories, colors, ylabel,
                          ylim=None, y_offset_text=1,
                          text_precision='.2f', figsize=(6, 5), title=None):
    fig = plt.figure(figsize=figsize)
    ax = Subplot(fig, 111)
    fig.add_subplot(ax)
    ax.axis["left"].set_axisline_style("-|>", size=1.5)
    ax.axis["bottom"].set_visible(False)
    ax.axis["top"].set_visible(False)
    ax.axis["right"].set_visible(False)
    ax.set_ylabel(ylabel, fontsize=11)
    bars = ax.bar(categories, data, color=colors, edgecolor='black', alpha=0.8, width=0.6)
    if ylim is not None:
        ax.set_ylim(ylim)
    else:
        bottom = 0
        top = max(data) * 1.15 if max(data) > 0 else 10
        ax.set_ylim(bottom, top)
    ax.grid(axis='y', linestyle='--', alpha=0.6)
    ax.set_axisbelow(True)
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2., height + y_offset_text,
                f'{height:{text_precision}}%', ha='center', va='bottom',
                fontweight='bold', fontsize=10)
    transform = blended_transform_factory(ax.transData, ax.transAxes)
    for bar, cat in zip(bars, categories):
        x_center = bar.get_x() + bar.get_width() / 2.
        ax.text(x_center, -0.04, cat, transform=transform,
                ha='center', va='top', fontsize=10, fontweight='normal')
    xlim = ax.get_xlim()
    x_range = xlim[1] - xlim[0]
    ax.set_xlim(xlim[0] - 0.1 * x_range, xlim[1] + 0.2 * x_range)
    plt.subplots_adjust(bottom=0.2)
    plt.tight_layout(rect=[0, 0.1, 1, 0.95])
    if title:
        ax.set_title(title, fontsize=12, fontweight='bold')
    return fig, ax


def compute_weighted_metrics(df, suffix):
    total_speech = df[f'Total_Speech_{suffix}'].sum()
    fa_sum = df[f'FA_{suffix}'].sum()
    miss_sum = df[f'Miss_{suffix}'].sum()
    conf_sum = df[f'Conf_{suffix}'].sum()
    fa_pct = (fa_sum / total_speech) * 100
    miss_pct = (miss_sum / total_speech) * 100
    conf_pct = (conf_sum / total_speech) * 100
    der_pct = ((fa_sum + miss_sum + conf_sum) / total_speech) * 100
    purity_weighted = (df[f'Purity_{suffix}'] * df[f'Total_Speech_{suffix}']).sum() / total_speech * 100
    coverage_weighted = (df[f'Coverage_{suffix}'] * df[f'Total_Speech_{suffix}']).sum() / total_speech * 100
    return {
        'fa': fa_pct,
        'miss': miss_pct,
        'conf': conf_pct,
        'der': der_pct,
        'purity': purity_weighted,
        'coverage': coverage_weighted
    }


df_base = pd.read_csv("/content/drive/MyDrive/pyannote_results/pyannote_metrics_20260323_133442.csv")
configs = ["strict", "collar", "no_ovl", "clean"]
for cfg in configs:
    base = compute_weighted_metrics(df_base, cfg)
    plot_single_histogram(
        data=[base['miss'], base['fa'], base['conf']],
        categories=['Missed', 'FA', 'Confusion'],
        colors=['#FF6B6B', '#4D96FF', '#6BCB77'],
        ylabel='Доля ошибок (%)',
        ylim=(0, max(base['miss'], base['fa'], base['conf']) * 1.15),
        y_offset_text=0.05,
        figsize=(6, 5),
    )
    plt.savefig(f'base_der_components_{cfg}.png', dpi=300, bbox_inches='tight')
    plt.show()
    plot_single_histogram(
        data=[base['purity'], base['coverage']],
        categories=['Purity', 'Coverage'],
        colors=['#FFD93D', '#A084CA'],
        ylabel='Значение метрики (%)',
        ylim=(0, 105),
        y_offset_text=0.04,
        text_precision='.1f',
        figsize=(5, 5),
    )
    plt.savefig(f'base_purity_coverage_{cfg}.png', dpi=300, bbox_inches='tight')
    plt.show()